# Refusal-direction reproduction setup on Colab Pro

This notebook mirrors the project tasks for the first reproduction pass:

1. Verify a CUDA GPU runtime.
2. Clone/sync the project and install dependencies with `uv`.
3. Authenticate to Hugging Face for gated Llama 3.2 access.
4. Download/cache the Llama 3.2 Instruct models.
5. Prepare AdvBench harmful instructions and length-matched Alpaca benign instructions.
6. Verify the prompts use the **actual Llama 3.2 chat template**.
7. Run a short smoke test on a few prepared dataset prompts.

Before running, choose **Runtime → Change runtime type → Hardware accelerator → GPU**.

## Configuration

Adjust these values before running the notebook if needed. The default dataset size matches the project task (`mise prepare-datasets`).

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/cayblood/refusal_direction_extended.git"
PROJECT_DIR = Path("/content/refusal_direction_extended")

MODEL_1B = "meta-llama/Llama-3.2-1B-Instruct"
MODEL_3B = "meta-llama/Llama-3.2-3B-Instruct"
DATASET_DIR = Path("data/refusal_datasets")
DATASET_SAMPLE_SIZE = 256
DATASET_SEED = 0

# Smoke test settings. Keep these small for quick validation.
SMOKE_MODEL = MODEL_1B
SMOKE_EXAMPLES_PER_CLASS = 2
SMOKE_MAX_NEW_TOKENS = 48

## Verify GPU runtime

If this cell fails, switch the runtime to a GPU runtime before continuing.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "CUDA is unavailable. In Colab, choose Runtime → Change runtime type → GPU."
)
print(torch.cuda.get_device_name(0))

## Get the project code

If you opened this notebook from the repository in Colab, the checkout may already exist. Otherwise, this clones `REPO_URL`.

In [ ]:
if PROJECT_DIR.exists():
    print(f"Using existing checkout: {PROJECT_DIR}")
    %cd {PROJECT_DIR}
    !git pull --ff-only
else:
    if not REPO_URL:
        raise ValueError(
            "Set REPO_URL to your repository URL, then rerun this cell."
        )
    !git clone {REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}

## Install project tooling and dependencies

Colab runtimes are ephemeral, so this installs `uv` and syncs the project dependencies into the runtime.

In [ ]:
!pip install -q uv
!uv sync

## Hugging Face authentication

Llama 3.2 Instruct is gated. AdvBench may also require accepting the dataset agreement. This cell prompts securely and stores the token only in the current Colab runtime environment. Do not paste tokens directly into saved notebook cells.

In [ ]:
import os
import subprocess
from getpass import getpass

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Hugging Face token: ")

subprocess.run(
    ["uv", "run", "hf", "auth", "login", "--token", os.environ["HF_TOKEN"]],
    check=True,
)
subprocess.run(["uv", "run", "hf", "auth", "whoami"], check=True)

## Verify project environment

This uses `uv run` so it checks the project environment, not only the notebook kernel.

In [ ]:
import subprocess
import textwrap

code = textwrap.dedent(
    """
    from importlib.metadata import version

    import torch

    print("torch", version("torch"))
    print("transformer-lens", version("transformer-lens"))
    print("transformers", version("transformers"))
    print("datasets", version("datasets"))
    print("cuda", torch.cuda.is_available())
    print("gpu", torch.cuda.get_device_name(0))
    """
)

subprocess.run(["uv", "run", "python", "-c", code], check=True)

## Download/cache models

This downloads both Llama 3.2 Instruct models into the active Colab runtime cache. If you want persistent caching across Colab sessions, mount Drive and set `HF_HOME` before running this cell.

In [ ]:
!uv run hf download {MODEL_1B} --repo-type model
!uv run hf download {MODEL_3B} --repo-type model

## Prepare reproduction datasets

This mirrors:

```sh
mise prepare-datasets
```

It prepares ~256 harmful AdvBench instructions and ~256 length-matched benign Alpaca instructions, formatted with the Llama 3.2 chat template.

In [ ]:
!uv run python scripts/prepare_datasets.py \
  --model {MODEL_1B} \
  --sample-size {DATASET_SAMPLE_SIZE} \
  --seed {DATASET_SEED} \
  --output-dir {DATASET_DIR}

## Inspect and validate formatted prompts

This verifies that every prepared prompt has the expected Llama 3.2 chat-template structure and includes the assistant generation prompt. Wrong chat templates produce wrong activation positions, so this check is intentionally strict.

In [ ]:
import json
from pathlib import Path

paths = [DATASET_DIR / "harmful.jsonl", DATASET_DIR / "benign.jsonl"]
for path in paths:
    rows = [json.loads(line) for line in path.read_text().splitlines()]
    bad = []
    for index, record in enumerate(rows):
        prompt = record["formatted_prompt"]
        expected_prefix = (
            "<|begin_of_text|><|start_header_id|>system<|end_header_id|>"
        )
        expected_suffix = "<|start_header_id|>assistant<|end_header_id|>\n\n"
        checks = [
            prompt.startswith(expected_prefix),
            "<|start_header_id|>user<|end_header_id|>" in prompt,
            prompt.endswith(expected_suffix),
            record["instruction"] in prompt,
        ]
        if not all(checks):
            bad.append((index, checks))
    print(path, "rows=", len(rows), "bad=", len(bad))
    assert not bad, bad[:5]

first_path = DATASET_DIR / "harmful.jsonl"
first = json.loads(first_path.read_text().splitlines()[0])
print("First harmful raw instruction:")
print(first["instruction"])
print("\nFirst harmful formatted prompt prefix:")
print(repr(first["formatted_prompt"][:500]))

## Dataset smoke test on a few prepared inputs

This loads the smaller Llama 3.2 1B Instruct model with TransformerLens and generates short completions from the already-formatted dataset prompts. This validates the prepared files, tokenizer formatting, model loading, and generation path without running the full experiment.

In [ ]:
import gc
import json
import time
from pathlib import Path

import torch
from transformer_lens import HookedTransformer

smoke_records = []
for name in ["harmful", "benign"]:
    path = DATASET_DIR / f"{name}.jsonl"
    rows = [json.loads(line) for line in path.read_text().splitlines()]
    smoke_records.extend(rows[:SMOKE_EXAMPLES_PER_CLASS])

print(f"Loading {SMOKE_MODEL} for smoke test...")
model = HookedTransformer.from_pretrained_no_processing(
    SMOKE_MODEL,
    device="cuda",
    dtype=torch.float16,
    default_prepend_bos=False,
)

torch.set_grad_enabled(False)
for record in smoke_records:
    print("\n" + "=" * 80)
    label = record["label"].upper()
    pair_id = record["pair_id"]
    raw_tokens = record["raw_token_count"]
    print(f"{label} pair_id={pair_id} raw_tokens={raw_tokens}")
    print("Instruction:", record["instruction"])
    start = time.monotonic()
    generated = model.generate(
        record["formatted_prompt"],
        max_new_tokens=SMOKE_MAX_NEW_TOKENS,
        do_sample=False,
        stop_at_eos=True,
        prepend_bos=False,
        verbose=False,
    )
    prompt_length = len(record["formatted_prompt"])
    completion = generated[prompt_length:].strip()
    print(f"Completed in {time.monotonic() - start:.1f}s")
    print("Completion:")
    print(completion)

del model
gc.collect()
torch.cuda.empty_cache()

## Optional: run the original baseline sanity prompts

This mirrors:

```sh
mise baseline
```

It uses the hard-coded harmful/benign sanity prompts in `scripts/baseline.py`, not the prepared dataset files.

In [ ]:
# Full default baseline over both models.
# !uv run python scripts/baseline.py --device cuda

# Faster variant: only the smaller model and shorter completions.
# !uv run python scripts/baseline.py --device cuda \
#   --model {MODEL_1B} \
#   --max-new-tokens 64

## Useful variants

You can rerun dataset preparation with a smaller sample while debugging:

```sh
!uv run python scripts/prepare_datasets.py --sample-size 16 --output-dir data/refusal_datasets_debug
```

If AdvBench Hugging Face access is unavailable, the script automatically falls back to the canonical AdvBench CSV used by `llm-attacks`.